## Imports & Variables 

In [184]:
import pandas as pd
import pygeohash as pgh
import datetime
import sys
import os
import requests
from io import StringIO
from datetime import datetime, time 
from pathlib import Path

root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from python.kp_index import update_kp_data
from python.geo_location import create_geohashes



## Data Cleaning & Enrichment for US UAP Reports 1940 - 2014 

### Read the UAP Dataset into a Pandas Dataframe
- "../data/raw/uap_original_dataset.csv"
- a few rows in this large dataset have an extra column of irrelevent data
    - the fix: usecols=range(0, 11)

In [185]:
uap_df = pd.read_csv("../data/raw/uap_original_dataset.csv", usecols=range(0, 11), low_memory=False) 

uap_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88875 entries, 0 to 88874
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              88875 non-null  object 
 1   city                  88679 non-null  object 
 2   state                 81356 non-null  object 
 3   country               76314 non-null  object 
 4   shape                 85757 non-null  object 
 5   duration (seconds)    88873 non-null  object 
 6   duration (hours/min)  85772 non-null  object 
 7   comments              88749 non-null  object 
 8   date posted           88875 non-null  object 
 9   latitude              88875 non-null  object 
 10  longitude             88875 non-null  float64
dtypes: float64(1), object(10)
memory usage: 7.5+ MB


In [186]:
uap_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611


In [187]:
# At least one row in the 'datetime' col contains an invalid 24:00 for the time
# AI Use: Grok assistance in fixing any entries with 24:00 in formating datetime

# Make a copy of the original column
col = uap_df['datetime'].astype(str)

# Handle 24:00 cases
mask_24 = col.str.contains('24:00', na=False)
col = col.str.replace('24:00', '00:00', regex=False)

# Convert to real datetime
uap_df['datetime'] = pd.to_datetime(col, format='%m/%d/%Y %H:%M', errors='coerce')

# Fix the date rollover for 24:00
uap_df.loc[mask_24, 'datetime'] = uap_df.loc[mask_24, 'datetime'] + pd.Timedelta(days=1)

# Create the columns 
uap_df['datetime_formatted'] = uap_df['datetime'].dt.strftime('%Y-%m-%d %H:%M')   # yyyy-mm-dd hh:mm
uap_df['full_date']           = uap_df['datetime'].dt.strftime('%Y-%m-%d')         # yyyy-mm-dd only

### Create Dataframe with Only US sightings 

In [188]:
us_uap_df = uap_df[uap_df["country"].str.lower() == "us"].copy()
us_uap_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,20 minutes,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333,1965-10-10 23:45,1965-10-10


In [189]:
us_uap_after_1940_df = us_uap_df[
    us_uap_df["datetime"] >= '1940-01-01'
    
].copy()

us_uap_after_1940_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70276 entries, 0 to 88874
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              70276 non-null  datetime64[ns]
 1   city                  70276 non-null  object        
 2   state                 70276 non-null  object        
 3   country               70276 non-null  object        
 4   shape                 68047 non-null  object        
 5   duration (seconds)    70275 non-null  object        
 6   duration (hours/min)  68054 non-null  object        
 7   comments              70248 non-null  object        
 8   date posted           70276 non-null  object        
 9   latitude              70276 non-null  object        
 10  longitude             70276 non-null  float64       
 11  datetime_formatted    70276 non-null  object        
 12  full_date             70276 non-null  object        
dtypes: datetime64[ns](1),

### Find out how many null values remain

In [190]:
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                   2229
duration (seconds)         1
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

### Fix null values in 'shape' column
- begin with finding unique values 

In [191]:
us_uap_after_1940_df['shape'].unique()

array(['cylinder', 'circle', 'light', 'sphere', 'disk', 'fireball',
       'unknown', 'oval', 'other', 'rectangle', 'chevron', 'formation',
       'triangle', 'cigar', nan, 'delta', 'changing', 'diamond', 'flash',
       'egg', 'teardrop', 'cone', 'cross', 'pyramid', 'round', 'flare',
       'hexagon', 'crescent', 'changed'], dtype=object)

### Assign 'shape' NA and Null values as 'unknown'

In [192]:
us_uap_after_1940_df['shape'] = us_uap_after_1940_df['shape'].fillna('unknown')
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                      0
duration (seconds)         1
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

### fix durration (sec) null value - only one
- look at the row and decide to remove entry or fill in value .. 

In [193]:
us_uap_after_1940_df[us_uap_after_1940_df['duration (seconds)'].isnull()]

# airport sighting, unclear durration based on other values - drop row in place 
us_uap_after_1940_df.dropna(subset=['duration (seconds)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                      0
duration (seconds)         0
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

### remove 'duration hours / mins' 
- (null and wierd values + seconds column gives us duration and is more nomalized)

In [194]:
us_uap_after_1940_df.drop(columns = ['duration (hours/min)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

datetime               0
city                   0
state                  0
country                0
shape                  0
duration (seconds)     0
comments              28
date posted            0
latitude               0
longitude              0
datetime_formatted     0
full_date              0
dtype: int64

### Fix null values in 'comments' column

In [195]:
# what do the rows look like? 
us_uap_after_1940_df[us_uap_after_1940_df['comments'].isnull()]

# fill with null comments with 'no comment' 
us_uap_after_1940_df['comments'] = us_uap_after_1940_df['comments'].fillna('no comments')

us_uap_after_1940_df.isnull().sum()

datetime              0
city                  0
state                 0
country               0
shape                 0
duration (seconds)    0
comments              0
date posted           0
latitude              0
longitude             0
datetime_formatted    0
full_date             0
dtype: int64

### convert 'duration (seconds)' to int
- some entries have non-integer values in them
- use regex in conversion to remove non-integers 

In [196]:
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(float)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].round(0)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(int)
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333,1965-10-10 23:45,1965-10-10


### Convert latitude to float type (longitude is already a float)

In [197]:
us_uap_after_1940_df['latitude'] = us_uap_after_1940_df['latitude'].astype(float)

us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration (seconds)             int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date                     object
dtype: object

### Convert 'full_date' to datetime + create 'year' and 'month' column

In [198]:
us_uap_after_1940_df['full_date'] = pd.to_datetime(us_uap_after_1940_df['full_date'])

us_uap_after_1940_df['year'] = us_uap_after_1940_df['full_date'].astype(str).str[:4]
us_uap_after_1940_df['year'] = us_uap_after_1940_df['year'].astype(int)

us_uap_after_1940_df['month'] = us_uap_after_1940_df['full_date'].astype(str).str[5:7]
us_uap_after_1940_df['month'] = us_uap_after_1940_df['month'].astype(int)

us_uap_after_1940_df.head()
us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration (seconds)             int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date             datetime64[ns]
year                           int64
month                          int64
dtype: object

# Enrich data even more with a 'season' column
- create season_groups dictionary
- map to 'month' value in each row

In [199]:
season_groups = {
    '01': 'winter', 
    '02': 'winter', 
    '03': 'spring', 
    '04': 'spring', 
    '05': 'spring',
    '06': 'summer',
    '07': 'summer', 
    '08': 'summer',
    '09': 'fall', 
    '10': 'fall',
    '11': 'fall',
    '12': 'winter',
}

us_uap_after_1940_df['season'] = us_uap_after_1940_df['month'].astype(str).str.zfill(2).map(season_groups)

us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall


### Add KP Index Data ('solar_kp_index', 'solar_ap_index' columns)
- Call update_kp_data function in module
- Read csv file into pandas df
- Ensure datetime is datetime 
- KP data is in UTZ and updated every 3 hours: for a given day, find the the data that matched to 9pm 21:00 US Central Time
- Create a 'full_date' column in the kp_9pm_us_cst_df to merge on.
- Merge to full_date to us uap sightings

In [200]:
update_kp_data()

   year  month  day  hour_start  hour_end  decimal_day_start  decimal_day_end  \
0  1932      1    1         0.0       1.5              0.000           0.0625   
1  1932      1    1         3.0       4.5              0.125           0.1875   
2  1932      1    1         6.0       7.5              0.250           0.3125   
3  1932      1    1         9.0      10.5              0.375           0.4375   
4  1932      1    1        12.0      13.5              0.500           0.5625   

      kp  ap  flag            datetime  
0  3.333  18     1 1932-01-01 00:00:00  
1  2.667  12     1 1932-01-01 03:00:00  
2  2.333   9     1 1932-01-01 06:00:00  
3  2.667  12     1 1932-01-01 09:00:00  
4  3.333  18     1 1932-01-01 12:00:00  


,year,month,day,hour_start,hour_end,decimal_day_start,decimal_day_end,kp,ap,flag,datetime
0,1932,1,1,0.0,1.5,0.000,0.0625,3.333,18,1,1932-01-01 00:00:00
1,1932,1,1,3.0,4.5,0.125,0.1875,2.667,12,1,1932-01-01 03:00:00
2,1932,1,1,6.0,7.5,0.250,0.3125,2.333,9,1,1932-01-01 06:00:00
3,1932,1,1,9.0,10.5,0.375,0.4375,2.667,12,1,1932-01-01 09:00:00
4,1932,1,1,12.0,13.5,0.500,0.5625,3.333,18,1,1932-01-01 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...
276250,2026,7,17,6.0,7.5,34531.250,34531.3125,0.667,3,0,2026-07-17 06:00:00
276251,2026,7,17,9.0,10.5,34531.375,34531.4375,0.667,3,0,2026-07-17 09:00:00
276252,2026,7,17,12.0,13.5,34531.500,34531.5625,1.333,5,0,2026-07-17 12:00:00
276253,2026,7,17,15.0,16.5,34531.625,34531.6875,1.333,5,0,2026-07-17 15:00:00


In [201]:
kp_df = pd.read_csv("../data/processed/kp_index.csv", usecols=('datetime', 'kp', 'ap'))
kp_df.head(20)

,kp,ap,datetime
0,3.333,18,1932-01-01 00:00:00
1,2.667,12,1932-01-01 03:00:00
2,2.333,9,1932-01-01 06:00:00
3,2.667,12,1932-01-01 09:00:00
4,3.333,18,1932-01-01 12:00:00
5,2.667,12,1932-01-01 15:00:00
6,3.333,18,1932-01-01 18:00:00
7,3.333,18,1932-01-01 21:00:00
8,3.667,22,1932-01-02 00:00:00
9,3.667,22,1932-01-02 03:00:00


In [202]:
# ensure datetime column is a datetime type
kp_df['datetime'] = pd.to_datetime(kp_df['datetime'])
kp_df.dtypes

kp                 float64
ap                   int64
datetime    datetime64[ns]
dtype: object

In [203]:
# kp table includes values in utz (5-6 hours ahead of most US timezones)
# most uap sightings in the US happen late in the evening / close 9:00pm or 21:00 
# 03:00:00 or 3am UTZ would be the best match for most sightings .. which is 9pm central standard time 

# only 03:00 utz (it will have the next day's date, so we would need to convert to US time zome)
target_time = time(3, 0, 0)

# Filter for exactly 03:00 UTC
kp_3am_utz_df = kp_df[kp_df['datetime'].dt.time == target_time].copy()

# Convert to US Central Standard Time (CST = UTC-6)
kp_9pm_us_cst_df = kp_3am_utz_df.copy()
kp_9pm_us_cst_df['datetime'] = kp_9pm_us_cst_df['datetime'] - pd.Timedelta(hours=6)

kp_9pm_us_cst_df.head()



,kp,ap,datetime
1,2.667,12,1931-12-31 21:00:00
9,3.667,22,1932-01-01 21:00:00
17,3.333,18,1932-01-02 21:00:00
25,0.333,2,1932-01-03 21:00:00
33,0.000,0,1932-01-04 21:00:00


In [204]:
# Create a 'full_date' column in the kp_9pm_us_cst_df to merge on

kp_9pm_us_cst_df['full_date'] = kp_9pm_us_cst_df['datetime'].dt.strftime('%Y-%m-%d')
kp_9pm_us_cst_df['full_date'] = pd.to_datetime(kp_9pm_us_cst_df['full_date'])
kp_9pm_us_cst_df.head()

,kp,ap,datetime,full_date
1,2.667,12,1931-12-31 21:00:00,1931-12-31
9,3.667,22,1932-01-01 21:00:00,1932-01-01
17,3.333,18,1932-01-02 21:00:00,1932-01-02
25,0.333,2,1932-01-03 21:00:00,1932-01-03
33,0.000,0,1932-01-04 21:00:00,1932-01-04


In [205]:
# Merge to full_date to us uap sightings

us_uap_after_1940_df = pd.merge(us_uap_after_1940_df, kp_9pm_us_cst_df[['full_date','kp', 'ap']], on='full_date', how='left')

us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season,kp,ap
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall,2.667,12
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall,2.667,12
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall,3.667,22
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall,1.667,6
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall,0.333,2


### Additional Cleanup
- create state_code column in uppercase to match bigfoot data and ERD
- rename columns to match ERD

In [206]:
# create state_code column in uppercase to match bigfoot data and ERD

us_uap_after_1940_df['state_code'] = us_uap_after_1940_df['state'].str.upper()

# rename to match ERD

us_uap_after_1940_df = us_uap_after_1940_df.rename(
    columns={
        'duration (seconds)': 'duration_secs',
        'kp': 'solar_kp_index', 
        'ap': 'solar_ap_index'
    }
)

### Group Shapes - Enrich Data with 'shape_group' column
- There are many unique 'shapes' reported, but many are similar in 'type'
    - example: 'fireball' and 'flash' are both 'light' types
- Create a shape_groups dictionary
    - Missing or unknown values will be grouped in the 'other' category 
- Map to 'shape' column 

In [207]:

# group shapes by types 

shape_groups = {
    'circle': 'round', 'sphere': 'round', 'disk': 'round', 
    'oval': 'round', 'round': 'round', 'crescent': 'round', 'dome': 'round',
    'egg': 'round', 'teardrop': 'round',
    
    'triangle': 'triangle', 'delta': 'triangle', 'chevron': 'triangle', 
    'pyramid': 'triangle', 'diamond': 'triangle',
    
    'cylinder': 'cigar', 'cigar': 'cigar',
    
    'light': 'light', 'fireball': 'light', 'flash': 'light', 'flare': 'light',
    
    'changing': 'changing', 'changed': 'changing', 'formation': 'changing', 'cone': 'changing',
    'cross': 'changing', 'hexagon': 'changing', 'rectangle': 'changing',
    
    'other': 'other', 
    'nan': 'other',
    'unknown': 'other',
    '': 'other'
}

us_uap_after_1940_df['shape_group'] = us_uap_after_1940_df['shape'].map(shape_groups)

us_uap_after_1940_df['shape_group'].value_counts()

shape_group
light       20796
round       20253
other       12139
triangle     8809
changing     5442
cigar        2836
Name: count, dtype: int64

### Check head and save to CSV in data/processed/ folder 

In [208]:
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration_secs,comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season,solar_kp_index,solar_ap_index,state_code,shape_group
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall,2.667,12,TX,cigar
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall,2.667,12,TX,round
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall,3.667,22,HI,light
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall,1.667,6,TN,round
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall,0.333,2,CT,round


In [209]:
# save to a csv copy before adding geohash and weather data

us_uap_after_1940_df.to_csv("../data/processed/us_uap_1940_v1.csv", index=False)

### Additional Enrichment Before Saving to Final

In [210]:
# addional enrichment? 

### Finished? Save to 'data/final/' folder 

In [211]:
us_uap_after_1940_df.to_csv("../data/final/uap_reports.csv", index=False)

## Data Cleaning & Enrichment for Bigfoot Dataset(s)

### Read the first Bigfoot Dataset 
- '../data/raw/bfro_locations.csv'

In [212]:
pd.set_option('display.max_columns', None) 

bigfoot_df = pd.read_csv('../data/raw/bfro_locations.csv', index_col=False)

bigfoot_df.shape
bigfoot_df.describe(include='all')
bigfoot_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4250 entries, 0 to 4249
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   index           4250 non-null   int64  
 1   number          4250 non-null   int64  
 2   title           4250 non-null   object 
 3   classification  4250 non-null   object 
 4   timestamp       4250 non-null   object 
 5   latitude        4250 non-null   float64
 6   longitude       4250 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 232.6+ KB


In [213]:
bigfoot_df.isna().sum()

index             0
number            0
title             0
classification    0
timestamp         0
latitude          0
longitude         0
dtype: int64

### Rename 'number' to 'bf_id' 
- We will use this as the Primary Key later on
- Check datatypes 

In [214]:
bigfoot_df = bigfoot_df.rename(columns={"number": "bf_id"})
bigfoot_df.dtypes

index               int64
bf_id               int64
title              object
classification     object
timestamp          object
latitude          float64
longitude         float64
dtype: object

### Merge geohash data from the other bigfoot dataset 
- sample random entry to see if data matches
- rename 'number' to 'bf_id' in geohashed dataset
- merge on bf_id

In [215]:
bigfoot_df.loc[bigfoot_df['bf_id'] == 799]

,index,bf_id,title,classification,timestamp,latitude,longitude
13,13,799,Report 799: Person fishing recounts story of h...,Class A,1978-04-15T12:00:00Z,34.92855,-87.1105


In [216]:
# Read in geohashed dataset csv and ensure 799 is a match

bigfoot_geocoded_df = pd.read_csv('../data/raw/bfro_reports_geocoded.csv')
bigfoot_geocoded_df['number'] = bigfoot_geocoded_df['number'].astype(int)
bigfoot_geocoded_df.loc[bigfoot_geocoded_df['number'] == 799]

,index,observed,location_details,county,state,season,title,latitude,longitude,date,number,classification,geohash,temperature_high,temperature_mid,temperature_low,dew_point,humidity,cloud_cover,moon_phase,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed
1219,1219,"The main sighting, as I wish to refer to it he...",The main sighting report I leave here happened...,Limestone County,Alabama,Spring,Report 799: Person fishing recounts story of h...,34.92855,-87.1105,1978-04-15,799,Class A,dn4n9y81d1,76.72,59.2,41.68,42.34,0.55,0.56,0.26,0.0,0.0,NaN,1019.14,Mostly cloudy throughout the day.,5.0,9.91,40.0,2.63


In [217]:
# so, it looks like we will need to do a left join from the bigfoot_df to the bigfoot_geocoded_df ... bf_id = number
# We need the state and city and wx data from the geocoded - so we will go through with the merge

bigfoot_geocoded_df = bigfoot_geocoded_df.rename(columns={"number": "bf_id"})
combined_bigfoot_df = pd.merge(
    bigfoot_df, 
    bigfoot_geocoded_df[[
        'bf_id', 
        'date', 
        'season', 
        'state', 
        'geohash',  
        'temperature_mid', 
        'precip_type', 
        'dew_point', 
        'cloud_cover', 
        'moon_phase', 
        'observed']], 
        on='bf_id', 
        how='left'
    )

combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,precip_type,dew_point,cloud_cover,moon_phase,observed
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,NaN,rain,42.75,1.00,0.49,My hiking partner and I arrived late to the Ke...
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.595,rain,42.15,0.95,0.54,"I was going for a drive. I had 3 kids with me,..."
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.490,rain,41.51,0.98,0.62,This incident happened last night just after 1...
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.275,NaN,43.18,0.33,0.03,"My daughter and I were traveling to Tok, Alask..."
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.665,NaN,17.35,0.16,0.84,I and two of my friends were bored one night s...


In [218]:
combined_bigfoot_df.dtypes

index                int64
bf_id                int64
title               object
classification      object
timestamp           object
latitude           float64
longitude          float64
date                object
season              object
state               object
geohash             object
temperature_mid    float64
precip_type         object
dew_point          float64
cloud_cover        float64
moon_phase         float64
observed            object
dtype: object

In [219]:
# full_date(datetime), year(int), month(int), dat(int)

combined_bigfoot_df['full_date'] = pd.to_datetime(combined_bigfoot_df['date'], format='%Y-%m-%d', errors='coerce')


combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,precip_type,dew_point,cloud_cover,moon_phase,observed,full_date
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,NaN,rain,42.75,1.00,0.49,My hiking partner and I arrived late to the Ke...,2000-06-16
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.595,rain,42.15,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",1995-05-15
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.490,rain,41.51,0.98,0.62,This incident happened last night just after 1...,2004-02-09
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.275,NaN,43.18,0.33,0.03,"My daughter and I were traveling to Tok, Alask...",2004-06-18
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.665,NaN,17.35,0.16,0.84,I and two of my friends were bored one night s...,2004-02-15


In [220]:
combined_bigfoot_df['full_date'].describe()

count                             4045
mean     1999-06-22 13:15:17.577255808
min                1869-11-10 00:00:00
25%                1990-09-15 00:00:00
50%                2003-11-16 00:00:00
75%                2009-08-16 00:00:00
max                2021-11-27 00:00:00
Name: full_date, dtype: object

In [221]:
combined_bigfoot_df = combined_bigfoot_df[combined_bigfoot_df['full_date'] > '1949']
combined_bigfoot_df['full_date'].describe()

count                             4033
mean     1999-09-03 02:36:01.963798528
min                1949-08-15 00:00:00
25%                1990-10-31 00:00:00
50%                2003-11-29 00:00:00
75%                2009-08-25 00:00:00
max                2021-11-27 00:00:00
Name: full_date, dtype: object

In [222]:
combined_bigfoot_df.isna().sum()

index                 0
bf_id                 0
title                 0
classification        0
timestamp             0
latitude              0
longitude             0
date                  0
season                0
state                 0
geohash               0
temperature_mid     847
precip_type        2310
dew_point           660
cloud_cover         949
moon_phase          637
observed             31
full_date             0
dtype: int64

In [223]:
combined_bigfoot_df.drop(columns=['precip_type'], inplace=True)
combined_bigfoot_df.isna().sum()

index                0
bf_id                0
title                0
classification       0
timestamp            0
latitude             0
longitude            0
date                 0
season               0
state                0
geohash              0
temperature_mid    847
dew_point          660
cloud_cover        949
moon_phase         637
observed            31
full_date            0
dtype: int64

In [224]:
combined_bigfoot_df.dtypes

index                       int64
bf_id                       int64
title                      object
classification             object
timestamp                  object
latitude                  float64
longitude                 float64
date                       object
season                     object
state                      object
geohash                    object
temperature_mid           float64
dew_point                 float64
cloud_cover               float64
moon_phase                float64
observed                   object
full_date          datetime64[ns]
dtype: object

In [225]:
combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,dew_point,cloud_cover,moon_phase,observed,full_date
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,NaN,42.75,1.00,0.49,My hiking partner and I arrived late to the Ke...,2000-06-16
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.595,42.15,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",1995-05-15
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.490,41.51,0.98,0.62,This incident happened last night just after 1...,2004-02-09
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.275,43.18,0.33,0.03,"My daughter and I were traveling to Tok, Alask...",2004-06-18
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.665,17.35,0.16,0.84,I and two of my friends were bored one night s...,2004-02-15


In [226]:
combined_bigfoot_df['temperature_mid'] = combined_bigfoot_df['temperature_mid'].fillna(
    round(combined_bigfoot_df['temperature_mid'].mean(), 3))

# round the F temp down to 1 decimal point
combined_bigfoot_df['temperature_mid'] = round(combined_bigfoot_df['temperature_mid'], 1)

combined_bigfoot_df['dew_point'] = combined_bigfoot_df['dew_point'].fillna(
    round(combined_bigfoot_df['dew_point'].mean(), 2))

# round down to 1 decimal 
combined_bigfoot_df['dew_point'] = round(combined_bigfoot_df['dew_point'], 1)

combined_bigfoot_df['cloud_cover'] = combined_bigfoot_df['cloud_cover'].fillna(
    round(combined_bigfoot_df['cloud_cover'].mean(), 2))

combined_bigfoot_df['moon_phase'] = combined_bigfoot_df['moon_phase'].fillna(
    round(combined_bigfoot_df['moon_phase'].mean(), 2))

# fill in the observed with the title
combined_bigfoot_df['observed'] = combined_bigfoot_df['observed'].fillna(combined_bigfoot_df['title'])

In [227]:
combined_bigfoot_df.isna().sum()

index              0
bf_id              0
title              0
classification     0
timestamp          0
latitude           0
longitude          0
date               0
season             0
state              0
geohash            0
temperature_mid    0
dew_point          0
cloud_cover        0
moon_phase         0
observed           0
full_date          0
dtype: int64

In [228]:
combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,dew_point,cloud_cover,moon_phase,observed,full_date
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,57.8,42.8,1.00,0.49,My hiking partner and I arrived late to the Ke...,2000-06-16
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.6,42.2,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",1995-05-15
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.5,41.5,0.98,0.62,This incident happened last night just after 1...,2004-02-09
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.3,43.2,0.33,0.03,"My daughter and I were traveling to Tok, Alask...",2004-06-18
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.7,17.4,0.16,0.84,I and two of my friends were bored one night s...,2004-02-15


### Merge KP Index data - use kp_9pm_us_cst_df['full_date']

In [229]:
# left merge kp data for the date of each sighting 
combined_bigfoot_df = pd.merge(combined_bigfoot_df, kp_9pm_us_cst_df[['full_date','kp', 'ap']], on='full_date', how='left')

combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,dew_point,cloud_cover,moon_phase,observed,full_date,kp,ap
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,57.8,42.8,1.00,0.49,My hiking partner and I arrived late to the Ke...,2000-06-16,0.667,3
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.6,42.2,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",1995-05-15,4.000,27
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.5,41.5,0.98,0.62,This incident happened last night just after 1...,2004-02-09,1.333,5
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.3,43.2,0.33,0.03,"My daughter and I were traveling to Tok, Alask...",2004-06-18,1.667,6
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.7,17.4,0.16,0.84,I and two of my friends were bored one night s...,2004-02-15,1.333,5


In [230]:
# bigfoot geohash columns
 
combined_bigfoot_df['geohash_5'] = combined_bigfoot_df['geohash'].str[:5]
combined_bigfoot_df['geohash_6'] = combined_bigfoot_df['geohash'].str[:6]
combined_bigfoot_df['geohash_7'] = combined_bigfoot_df['geohash'].str[:7]

# check the head

combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,dew_point,cloud_cover,moon_phase,observed,full_date,kp,ap,geohash_5,geohash_6,geohash_7
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,57.8,42.8,1.00,0.49,My hiking partner and I arrived late to the Ke...,2000-06-16,0.667,3,bffmu,bffmu5,bffmu5x
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.6,42.2,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",1995-05-15,4.000,27,c1c9f,c1c9fn,c1c9fne
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.5,41.5,0.98,0.62,This incident happened last night just after 1...,2004-02-09,1.333,5,c1cd1,c1cd19,c1cd197
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.3,43.2,0.33,0.03,"My daughter and I were traveling to Tok, Alask...",2004-06-18,1.667,6,bg5q4,bg5q49,bg5q496
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.7,17.4,0.16,0.84,I and two of my friends were bored one night s...,2004-02-15,1.333,5,bdv7r,bdv7re,bdv7re9


### Mapping dictionary to create state_code column (unique id for state)
- grok4 used to generate dictionary 

In [231]:
state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC'
}

combined_bigfoot_df['state_code'] = combined_bigfoot_df['state'].map(state_to_code)
combined_bigfoot_df['state_code'] = combined_bigfoot_df['state_code'].fillna('Unknown')

combined_bigfoot_df['year']  = combined_bigfoot_df['full_date'].dt.year.astype(int)
combined_bigfoot_df['month'] = combined_bigfoot_df['full_date'].dt.month.astype(int)
combined_bigfoot_df['day']   = combined_bigfoot_df['full_date'].dt.day.astype(int)


combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,dew_point,cloud_cover,moon_phase,observed,full_date,kp,ap,geohash_5,geohash_6,geohash_7,state_code,year,month,day
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,57.8,42.8,1.00,0.49,My hiking partner and I arrived late to the Ke...,2000-06-16,0.667,3,bffmu,bffmu5,bffmu5x,AK,2000,6,16
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.6,42.2,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",1995-05-15,4.000,27,c1c9f,c1c9fn,c1c9fne,AK,1995,5,15
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.5,41.5,0.98,0.62,This incident happened last night just after 1...,2004-02-09,1.333,5,c1cd1,c1cd19,c1cd197,AK,2004,2,9
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.3,43.2,0.33,0.03,"My daughter and I were traveling to Tok, Alask...",2004-06-18,1.667,6,bg5q4,bg5q49,bg5q496,AK,2004,6,18
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.7,17.4,0.16,0.84,I and two of my friends were bored one night s...,2004-02-15,1.333,5,bdv7r,bdv7re,bdv7re9,AK,2004,2,15


In [232]:
combined_bigfoot_df['full_date'].describe()

count                             4033
mean     1999-09-03 02:36:01.963798528
min                1949-08-15 00:00:00
25%                1990-10-31 00:00:00
50%                2003-11-29 00:00:00
75%                2009-08-25 00:00:00
max                2021-11-27 00:00:00
Name: full_date, dtype: object

### check datatypes

In [233]:
combined_bigfoot_df.dtypes

index                       int64
bf_id                       int64
title                      object
classification             object
timestamp                  object
latitude                  float64
longitude                 float64
date                       object
season                     object
state                      object
geohash                    object
temperature_mid           float64
dew_point                 float64
cloud_cover               float64
moon_phase                float64
observed                   object
full_date          datetime64[ns]
kp                        float64
ap                          int64
geohash_5                  object
geohash_6                  object
geohash_7                  object
state_code                 object
year                        int64
month                       int64
day                         int64
dtype: object

### reorder columns, drop duplicate rows and write to csv in '../data/processed/'

In [234]:
combined_bigfoot_df = combined_bigfoot_df[[
    'bf_id', 'full_date', 'title', 'state_code', 'state', 
    'latitude', 'longitude', 'geohash_5', 'geohash_6', 'geohash_7', 'geohash', 
    'date', 'year', 'month', 'day', 'season',
    'temperature_mid', 'dew_point',  'cloud_cover', 'moon_phase',
    'classification',  'observed',
    'kp', 'ap'
]]

combined_bigfoot_df = combined_bigfoot_df.drop_duplicates(subset=['bf_id'])

combined_bigfoot_df.sort_values(by='bf_id', ascending=True, inplace=True)

combined_bigfoot_df = combined_bigfoot_df.rename(columns={
    'kp': 'solar_kp_index',
    'ap': 'solar_ap_index'
})

combined_bigfoot_df.to_csv("../data/processed/combined_bigfoot_v1.csv", index=False)

### save final version of bigfoot dataset to '../data/final/'

In [235]:
combined_bigfoot_df.to_csv("../data/final/bigfoot_reports.csv", index=False)